# PerovSeek：配方推荐、光谱预测与实测反馈
## PerovSeek: formulation recommendation, spectral prediction and measured feedback

本 notebook 将完整演示放在一个文件中。数据路径、参数、GP 构建、采集函数、数据划分、模型创建、推理及图表均直接写在下方；仅复用原仓库的模型架构和训练循环。新建 `workflow_helpers.py` 只将训练结果转换为可移植的 `state_dict`。

This single notebook exposes data paths, parameters, GP construction, acquisition, splits, model creation, inference and plots. It reuses the repository's architecture and training loop; the small helper only converts the locally trained checkpoint to a portable `state_dict`.

**范围 / Scope.** 1.59 eV 仓库示例，不是文章中 AIO 性能的复现。BO 配方记录和光谱样本来自不同工作簿；新候选尚未实测，也不与光谱测试例配对。/ This is a 1.59 eV repository example, not a reproduction of the manuscript's AIO performance. BO records and spectral samples are separate datasets; new candidates are unmeasured and are not paired with the spectral test examples.

**启动 / Start.** 将本文件、`workflow_helpers.py` 和 `requirements-demo.txt` 放在 PerovSeek 仓库根目录。使用 Python 3.12 环境执行 `python -m pip install -r requirements-demo.txt`，然后执行 `python -m notebook PerovSeek_workflow.ipynb`。选择 **Restart Kernel and Run All**。首次运行会真实训练 100 epochs（CPU 通常数分钟）；之后默认加载此 notebook 自己保存的 checkpoint。/ Put the three files in the repository root, install the requirements and run the notebook. The first complete run trains for 100 epochs; subsequent runs load its saved checkpoint.

原代码 / Source: [PerovSeek](https://github.com/HM-Laboratory/PerovSeek), `Bayesian optimization/Bayesian_Optimization_parallel.ipynb` and `Pre-trained model/Fusion.ipynb`.


### 1. 导入依赖
### 1. Import dependencies


In [ ]:
from pathlib import Path
import hashlib
import json
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch.utils.data import TensorDataset, DataLoader
from IPython.display import display
from botorch.models import SingleTaskGP
from botorch.models.transforms import Normalize, Standardize
from botorch.fit import fit_gpytorch_mll
from botorch.acquisition import qLogNoisyExpectedImprovement
from botorch.sampling.normal import SobolQMCNormalSampler
from botorch.optim import optimize_acqf
from gpytorch.kernels import MaternKernel, ScaleKernel
from gpytorch.mlls import ExactMarginalLogLikelihood


### 2. 仓库路径、随机种子与输出目录
### 2. Repository paths, random seed and output directory
所有路径相对于仓库；不依赖原制作电脑或外部结果文件。

All paths are relative to the repository; no original workstation or external result files are required.


In [ ]:
REPO = Path.cwd().resolve()
if not (REPO / "Pre-trained model").is_dir():
    REPO = REPO.parent  # Also permits launching from a direct child folder.
assert (REPO / "Pre-trained model" / "model.py").is_file(), "Open the repository root."
sys.path.insert(0, str(REPO / "Pre-trained model"))
sys.path.insert(0, str(REPO))
import model as spectral_models
from utility import load_ptmodel
from workflow_helpers import fine_train_portable

OUTPUT = REPO / "demo_outputs"
OUTPUT.mkdir(exist_ok=True)
SEED = 42
DEVICE = "cpu"
torch.set_num_threads(2)
torch.manual_seed(SEED)
np.random.seed(SEED)
plt.rcParams.update({"figure.dpi": 120, "font.size": 11, "axes.spines.top": False,
                     "axes.spines.right": False})
print("Repository:", REPO.name, "| device:", DEVICE, "| seed:", SEED)


### 3. 读取实测配方记录，明确优化目标
### 3. Read measured formulation records and choose the target
原 BO notebook 使用 `Voc*FF`；这里明确改用工作簿已有的实测 `PCE` 列作为演示配置。保留原始成分标签和数值单位。六个输入是 SAM/添加剂变量，溶剂比例已固定。

The source BO notebook uses `Voc*FF`; this demonstration explicitly selects the existing measured `PCE` column. Source component labels and value units are retained. The six inputs are SAM/additive variables with fixed solvent fractions.


In [ ]:
BO_FILE = REPO / "Bayesian optimization" / "data" / "BO_SAM_Additive.xlsx"
BO_SHEET = "init"
TARGET = "PCE"  # Change to "Voc*FF" to use the source notebook's target.
FEATURES = ["Me-4", "Py3", "4PADCB", "4F-BASZ", "F3EABr", "5F-SPBS"]
formulations = pd.read_excel(BO_FILE, sheet_name=BO_SHEET)
assert TARGET in {"PCE", "Voc*FF"}
assert not formulations[FEATURES + [TARGET]].isna().any().any()
train_X = torch.tensor(formulations[FEATURES].to_numpy(), dtype=torch.double)
train_Y = torch.tensor(formulations[[TARGET]].to_numpy(), dtype=torch.double)
display(formulations[["DMF", "NFM", "EA"]].drop_duplicates())
display(formulations[FEATURES + [TARGET]].head(6))
print(f"{len(formulations)} measured records; maximize {TARGET}.")


### 4. 显式设置搜索边界、约束和优化预算
### 4. Set bounds, constraints and optimization budget explicitly
原代码为 62 个候选、36 进程；这里为 6 个候选、单 CPU 进程，并减小重启和采样预算。保留原核、噪声标准差 2 和两组总量约束。这些是仓库示例参数，不代表文章所有阶段的配置。

The source uses 62 candidates and 36 processes. This demonstration uses six candidates on one CPU process with smaller restart/sampling budgets. The source kernel, noise standard deviation of 2 and two sum constraints are retained; these example settings do not represent every stage of the manuscript.


In [ ]:
BATCH_SIZE = 6
NOISE_SD = 2.0
MC_SAMPLES = 128
NUM_RESTARTS = 4
RAW_SAMPLES = 128
FIT_MAXITER = 40
OPT_MAXITER = 80
bounds = torch.tensor([[0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
                       [0.6, 0.7, 1.5, 0.5, 1.2, 1.5]], dtype=torch.double)
# BoTorch uses sum(coefficients * x[indices]) >= rhs.
constraints = [
    (torch.tensor([0, 1, 2]), torch.tensor([-1., -1., -1.], dtype=torch.double), -1.6),
    (torch.tensor([3, 4, 5]), torch.tensor([-1., -1., -1.], dtype=torch.double), -2.5),
]
bo_config = dict(target=TARGET, source_target="Voc*FF", features=FEATURES, seed=SEED,
                 batch_size=BATCH_SIZE, source_batch_size=62, noise_sd=NOISE_SD,
                 mc_samples=MC_SAMPLES, restarts=NUM_RESTARTS, raw_samples=RAW_SAMPLES,
                 fit_maxiter=FIT_MAXITER, opt_maxiter=OPT_MAXITER, processes=1)
display(pd.DataFrame(bounds.numpy().T, index=FEATURES, columns=["Lower", "Upper"]))
print("Me-4 + Py3 + 4PADCB <= 1.6; 4F-BASZ + F3EABr + 5F-SPBS <= 2.5")


### 5. 构造并拟合高斯过程
### 5. Construct and fit the Gaussian process


In [ ]:
torch.manual_seed(SEED)
gp = SingleTaskGP(
    train_X=train_X,
    train_Y=train_Y,
    train_Yvar=torch.full_like(train_Y, NOISE_SD ** 2),
    covar_module=ScaleKernel(MaternKernel(nu=1.5, ard_num_dims=len(FEATURES))),
    input_transform=Normalize(d=len(FEATURES)),
    outcome_transform=Standardize(m=1),
)
mll = ExactMarginalLogLikelihood(gp.likelihood, gp)
fit_gpytorch_mll(mll, optimizer_kwargs={"options": {"maxiter": FIT_MAXITER}})
lengthscales = gp.covar_module.base_kernel.lengthscale.detach().numpy().ravel()
display(pd.Series(lengthscales, index=FEATURES, name="Fitted length scale"))


### 6. 优化 qLogNEI，生成并导出候选
### 6. Optimize qLogNEI, generate and export candidates
这些是新提出、尚未实测的配方。GP 输出不是器件测量；若后验均值相近或相同，不能据此宣称明确排名或性能提升。

These are new, unmeasured formulations. GP outputs are not device measurements. Similar or identical posterior means do not establish a meaningful ranking or improvement.


In [ ]:
sampler = SobolQMCNormalSampler(sample_shape=torch.Size([MC_SAMPLES]), seed=SEED)
acquisition = qLogNoisyExpectedImprovement(model=gp, X_baseline=train_X, sampler=sampler)
candidates, acquisition_value = optimize_acqf(
    acq_function=acquisition, bounds=bounds, q=BATCH_SIZE,
    num_restarts=NUM_RESTARTS, raw_samples=RAW_SAMPLES,
    options={"batch_limit": 2, "maxiter": OPT_MAXITER}, inequality_constraints=constraints,
)
values = candidates.detach().numpy()
assert np.all(values >= bounds[0].numpy() - 1e-6) and np.all(values <= bounds[1].numpy() + 1e-6)
assert np.all(values[:, :3].sum(1) <= 1.6 + 1e-6) and np.all(values[:, 3:].sum(1) <= 2.5 + 1e-6)
with torch.no_grad():
    posterior = gp.posterior(candidates)
recommended = pd.DataFrame(values, columns=FEATURES)
recommended.insert(0, "candidate_id", [f"D{i+1:02d}" for i in range(BATCH_SIZE)])
recommended[f"gp_mean_{TARGET}"] = posterior.mean.numpy().ravel()
recommended[f"gp_sd_{TARGET}"] = posterior.variance.sqrt().numpy().ravel()
recommended.to_csv(OUTPUT / "unmeasured_recommendations.csv", index=False)
(OUTPUT / "bo_config.json").write_text(json.dumps(bo_config, indent=2), encoding="utf-8")
display(recommended.round(4))
print("Posterior mean range:", float(np.ptp(posterior.mean.numpy())), "| Candidates remain unmeasured.")


### 7. 读取独立的 1.59 eV 光谱示例工作簿
### 7. Read the separate 1.59 eV spectral example workbook
下面的数据不对应刚生成的 D01–D06。完整工作簿包含 2268 条光谱及标签；不使用原小样例中光谱和标签数量不一致的文件。`PL_top`/`PL_bottom` 是完整工作簿的实际工作表名。

These data are not paired with D01–D06. The full workbook contains 2268 spectra and labels; the smaller example with mismatched counts is not used. `PL_top` and `PL_bottom` are the actual full-workbook sheet names.


In [ ]:
SPECTRAL_FILE = REPO / "data" / "1.59eV_additive_data.xlsx"
tables = pd.read_excel(SPECTRAL_FILE, sheet_name=["Abs", "PL_top", "PL_bottom", "Param"])
abs_table = tables["Abs"]
pl_top_table = tables["PL_top"]
pl_bottom_table = tables["PL_bottom"]
parameters = tables["Param"]
PCE_LABEL = "PCE#rs1"
pce_measured = parameters[PCE_LABEL].to_numpy(dtype=float)
sample_ids = parameters.iloc[:, 0].astype(str).to_numpy()
assert abs_table.shape[1] - 1 == pl_top_table.shape[1] - 1 == len(pce_measured)
assert pl_bottom_table.shape[1] - 1 == len(pce_measured)
display(parameters[[PCE_LABEL]].head())
print(f"{len(pce_measured)} spectra and labels from {SPECTRAL_FILE.name}")


### 8. 直接展示光谱归一化与波长通道
### 8. Normalize spectra and construct the wavelength channel explicitly
沿用原代码：Abs ÷ 4、PL 顶部 ÷ 120000、PL 底部 ÷ 150000；吸收选取 402–1000 nm 的 300 点，两个 PL 各 66 点；第二通道为相对 400 nm 的波长偏移。

Source preprocessing is retained: Abs ÷ 4, top PL ÷ 120000, bottom PL ÷ 150000; 300 absorption points at 402–1000 nm and 66 points for each PL trace. The second channel is wavelength minus 400 nm.


In [ ]:
abs_wavelength = abs_table.iloc[1:, 0].to_numpy(dtype=float)
pl_wavelength = pl_top_table.iloc[:66, 0].to_numpy(dtype=float)
abs_signal = abs_table.iloc[1:, 1:].to_numpy(dtype=float).T / 4
pl_top_signal = pl_top_table.iloc[:66, 1:].to_numpy(dtype=float).T / 120000
pl_bottom_signal = pl_bottom_table.iloc[:66, 1:].to_numpy(dtype=float).T / 150000
assert np.array_equal(abs_wavelength, np.arange(402, 1002, 2))
assert np.array_equal(pl_wavelength, pl_bottom_table.iloc[:66, 0].to_numpy(dtype=float))
signal = np.concatenate([abs_signal, pl_top_signal, pl_bottom_signal], axis=1)
wave_offset = np.concatenate([abs_wavelength, pl_wavelength, pl_wavelength]) - 400
spectrum = np.stack([signal, np.broadcast_to(wave_offset, signal.shape)], axis=1)
assert spectrum.shape == (len(pce_measured), 2, 432)
assert np.isfinite(spectrum).all() and np.isfinite(pce_measured).all()
print("Model input shape:", spectrum.shape, "= samples × channels × spectral points")


### 9. 绘制实际测量的光谱
### 9. Plot measured spectra


In [ ]:
SHOW_SAMPLE = 0
fig, axes = plt.subplots(1, 3, figsize=(13, 3.2), layout="constrained")
spectral_views = [(abs_wavelength, abs_signal, "Absorption"),
                  (pl_wavelength, pl_top_signal, "PL, top excitation"),
                  (pl_wavelength, pl_bottom_signal, "PL, bottom excitation")]
for ax, (wave, measured, title) in zip(axes, spectral_views):
    ax.plot(wave, measured[:12].T, color="#147d92", alpha=0.15, lw=1)
    ax.plot(wave, measured[SHOW_SAMPLE], color="#147d92", lw=2)
    ax.set(title=title, xlabel="Wavelength (nm)", ylabel="Normalized signal")
fig.savefig(OUTPUT / "measured_spectra.png", dpi=180)
plt.show()
print("Highlighted repository sample:", sample_ids[SHOW_SAMPLE], "| source row:", SHOW_SAMPLE)


### 10. 显式划分数据并选择训练参数
### 10. Split the data and choose training settings explicitly
沿用原代码的随机划分：500 训练、1500 测试，其余 268 验证。它只保证本次监督训练的集合不相交；预训练数据覆盖关系未知，不称为完全独立验证。首次运行自动训练；将 `RUN_TRAINING` 改为 `True` 可强制重新训练。

The source random split uses 500 training and 1500 test examples, leaving 268 for validation. It establishes disjoint sets for this supervised run; pretraining coverage is unknown, so this is not claimed as fully independent validation. The first run trains automatically; set `RUN_TRAINING = True` to retrain.


In [ ]:
TRAIN_COUNT, TEST_COUNT = 500, 1500
np.random.seed(SEED)
indices = np.arange(len(pce_measured))
np.random.shuffle(indices)
train_indices = indices[:TRAIN_COUNT]
validation_indices = indices[TRAIN_COUNT:-TEST_COUNT]
test_indices = indices[-TEST_COUNT:]
assert len(validation_indices) > 0
assert not (set(train_indices) & set(test_indices) or set(validation_indices) & set(test_indices))
splits = {"train": train_indices.tolist(), "validation": validation_indices.tolist(),
          "test": test_indices.tolist()}
(OUTPUT / "spectral_splits.json").write_text(json.dumps(splits), encoding="utf-8")
TRAINING = dict(batch_size=32, lr=1e-3, min_lr=1e-4, epochs=100,
                weight_decay=1e-4, warmup_epochs=0, device=DEVICE, delta=0.15)
PCE_SCALE = 25.0
CHECKPOINT = OUTPUT / "spectral_pce_state.pt"
RUN_TRAINING = not CHECKPOINT.exists()
print({name: len(rows) for name, rows in splits.items()}, "| train now:", RUN_TRAINING)
display(pd.Series(TRAINING, name="Training configuration"))


### 11. 创建数据加载器、预训练编码器及 PCE 预测头
### 11. Create data loaders, the pretrained encoder and the PCE prediction head
仓库分发的 checkpoint 是光谱重建/编码器权重，不是已训练的 PCE 预测器。下方创建新的预测头，并按原代码解冻末 10 个编码器块。原训练循环实际使用归一化 PCE 的平方误差之和；`delta` 在原参数中保留，但原循环未使用 Huber loss。

The distributed checkpoint contains the spectral reconstruction/encoder weights, not a trained PCE predictor. A new prediction head is created and the last ten encoder blocks are unfrozen. The original loop uses summed squared error on scaled PCE; its retained `delta` parameter does not activate the unused Huber loss.


In [ ]:
spectrum_tensor = torch.tensor(spectrum, dtype=torch.float32)
pce_tensor = torch.tensor(pce_measured[:, None] / PCE_SCALE, dtype=torch.float32)
train_dataset = TensorDataset(spectrum_tensor[train_indices], pce_tensor[train_indices])
validation_dataset = TensorDataset(spectrum_tensor[validation_indices], pce_tensor[validation_indices])
train_loader = DataLoader(train_dataset, batch_size=TRAINING["batch_size"], shuffle=True, num_workers=0)
validation_loader = DataLoader(validation_dataset, batch_size=TRAINING["batch_size"], shuffle=False, num_workers=0)

torch.manual_seed(SEED)  # Reset after BO for reproducible spectral-head initialization.
PRETRAINED = REPO / "Pre-trained model" / "Pre-trained_model.pth"
encoder = load_ptmodel(str(PRETRAINED), DEVICE)
ARCHITECTURE = dict(hidden_dim=[18], n=10, embed_dim=36, dropout_rate=None)
pce_model = spectral_models.FineTuneModel(encoder, **ARCHITECTURE).to(DEVICE)
optimizer = torch.optim.AdamW(pce_model.parameters(), lr=TRAINING["lr"],
                             weight_decay=TRAINING["weight_decay"])
print("Trainable parameters:", sum(p.numel() for p in pce_model.parameters() if p.requires_grad))


### 12. 训练或加载可移植的最佳验证模型
### 12. Train or load the portable best-validation checkpoint
helper 仅调用原 `utility.fine_train` 并转换 checkpoint 格式；训练过程、损失和学习率安排来自原代码。最终文件保存 `state_dict`、架构、归一化、数据划分和训练配置，不依赖整模型 pickle 或其他电脑的绝对路径。

The helper calls the original `utility.fine_train` and converts the checkpoint format. The training loop, loss and learning-rate schedule remain in the original source. The final file stores a `state_dict`, architecture, preprocessing, splits and settings, without depending on a whole-model pickle or workstation-specific paths.


In [ ]:
checkpoint_metadata = dict(
    dataset=str(SPECTRAL_FILE.relative_to(REPO)), label=PCE_LABEL,
    data_sha256=hashlib.sha256(SPECTRAL_FILE.read_bytes()).hexdigest(),
    pretrained_sha256=hashlib.sha256(PRETRAINED.read_bytes()).hexdigest(),
    architecture=ARCHITECTURE, training=TRAINING, seed=SEED, splits=splits,
    preprocessing={"abs_divisor": 4, "pl_top_divisor": 120000,
                   "pl_bottom_divisor": 150000, "wavelength_offset_nm": 400,
                   "pce_scale": PCE_SCALE},
)
if RUN_TRAINING:
    fine_train_portable(pce_model, optimizer, train_loader, validation_loader,
                        TRAINING, CHECKPOINT, checkpoint_metadata)
payload = torch.load(CHECKPOINT, map_location=DEVICE, weights_only=True)
assert payload["metadata"] == checkpoint_metadata, "Checkpoint configuration differs; retrain or restore settings."
pce_model.load_state_dict(payload["state_dict"])
pce_model.eval()
history = pd.DataFrame(payload["history"])
history.to_csv(OUTPUT / "training_history.csv", index=False)
display(history.tail())
print("Loaded best validation epoch:", payload["best_epoch"], "| checkpoint:", CHECKPOINT.name)


### 13. 直接执行测试集推理，保存原始预测与误差
### 13. Run test inference directly and save raw predictions and errors
所有模型输出原样保留，包括可能的负预测。MAE/RMSE 单位为百分点；这些是本次仓库示例结果，不是文章报告的准确率。

All model outputs are retained, including possible negative predictions. MAE/RMSE are in percentage points; these are this repository example's results, not the manuscript's reported accuracy.


In [ ]:
predicted_batches = []
with torch.inference_mode():
    for batch in spectrum_tensor[test_indices].split(128):
        predicted_scaled, _ = pce_model(batch.to(DEVICE))
        predicted_batches.append(predicted_scaled.cpu().numpy().ravel() * PCE_SCALE)
predicted_pce = np.concatenate(predicted_batches)
test_pce = pce_tensor[test_indices].numpy().ravel() * PCE_SCALE
error = predicted_pce - test_pce
test_results = pd.DataFrame({"sample_id": sample_ids[test_indices], "source_row": test_indices,
                             "measured_pce_percent": test_pce, "predicted_pce_percent": predicted_pce,
                             "error_percentage_points": error})
assert np.isfinite(test_results.select_dtypes("number").to_numpy()).all()
test_results.to_csv(OUTPUT / "spectral_test_predictions.csv", index=False)
metrics = dict(test_count=len(test_results), mae_pp=float(np.abs(error).mean()),
               rmse_pp=float(np.sqrt(np.mean(error ** 2))),
               r2=float(1 - np.sum(error ** 2) / np.sum((test_pce - test_pce.mean()) ** 2)),
               negative_predictions=int(np.sum(predicted_pce < 0)))
(OUTPUT / "spectral_metrics.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")
display(pd.Series(metrics, name="1.59 eV repository example"))


### 14. 完整显示预测与实测散点图
### 14. Plot predicted versus measured PCE without clipping outputs


In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 5.4), layout="constrained")
ax.scatter(test_pce, predicted_pce, s=13, alpha=0.4, color="#147d92", edgecolor="none")
ax.plot([0, 25], [0, 25], "--", color="gray", lw=1)
xmin, xmax = min(0, np.floor(test_pce.min()) - 1), max(25, np.ceil(test_pce.max()) + 1)
ymin, ymax = min(0, np.floor(predicted_pce.min()) - 1), max(25, np.ceil(predicted_pce.max()) + 1)
ax.set(xlabel="Measured PCE (%)", ylabel="Predicted PCE (%)", xlim=(xmin, xmax), ylim=(ymin, ymax),
       title="1.59 eV repository example")
ax.text(0.04, 0.96, f"n = {len(test_results)}\nMAE = {metrics['mae_pp']:.2f} percentage points",
        transform=ax.transAxes, va="top")
fig.savefig(OUTPUT / "spectral_pce_scatter.png", dpi=180)
plt.show()
print(metrics["negative_predictions"], "negative predictions retained without clipping.")


### 15. 对固定的已有测试例进行排序
### 15. Rank a fixed set of existing test examples
先固定测试集中的前 8 个例子，再按预测 PCE 排序；它们不是 BO 新候选。实测列是工作簿已有标签，既未回填为新实验，也未用于选择这 8 个样本。样本 ID 可能重复，因此同时保留原始行号。

The first eight test examples are fixed before sorting by predicted PCE. They are not the new BO candidates. The measured values are existing workbook labels, not newly acquired feedback or a selection criterion. Source row indices distinguish potentially repeated sample IDs.


In [ ]:
example_rows = test_results.iloc[:8].copy()
ranked_examples = example_rows.sort_values("predicted_pce_percent", ascending=False)
ranked_examples.insert(0, "prediction_rank", np.arange(1, len(ranked_examples) + 1))
ranked_examples.to_csv(OUTPUT / "ranked_existing_test_examples.csv", index=False)
display(ranked_examples[["prediction_rank", "sample_id", "source_row",
                         "predicted_pce_percent", "measured_pce_percent"]].round(2))
print("Ranking existing spectral examples only; D01–D06 remain unmeasured.")


### 16. 定义真实实验后才能填写的反馈表
### 16. Define feedback that can be completed only after real experiments
下一步应实际制备并测量器件，然后将已测得的配方–PCE 记录添加到数据集中再运行 BO。这里仅导出空白测量字段，不填入 GP 或光谱预测，也不宣称完成了新实验循环。

The next step is to fabricate and measure devices, then append measured formulation–PCE records before rerunning BO. This cell exports empty measurement fields. Neither GP nor spectral predictions are inserted as experimental feedback, and no new experimental cycle is claimed.


In [ ]:
feedback_template = recommended[["candidate_id"] + FEATURES].copy()
feedback_field = "measured_PCE_percent" if TARGET == "PCE" else "measured_Voc_times_FF"
feedback_template[feedback_field] = np.nan
feedback_template["measurement_status"] = "not measured"
feedback_template.to_csv(OUTPUT / "pending_experimental_feedback.csv", index=False)
display(feedback_template)
print("Saved inputs, configurations, raw predictions, plots and checkpoint under:", OUTPUT.name)
print("No new experimental measurements have been generated by this notebook.")
